# RSNA Knee V16 GPU Pilot
Private first-fold training. No score claim.
224 px, 8 centers, up to 3 series; eight epochs. Temporary image cache; saved epoch checkpoints.
Stability fix: FP32 sequence head, bounded same-batch AMP retries, GPU preflight and batch progress logs.

In [ ]:
import subprocess, sys, os
os.environ['USE_TF']='0'
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
    'pydicom==3.0.2','pylibjpeg>=2,<3','pylibjpeg-libjpeg>=2,<3',
    'pylibjpeg-openjpeg>=2,<3','transformers==4.57.6'])
import torch
assert torch.cuda.is_available(), 'GPU allocation required; refusing CPU training'
torch.set_num_threads(2)
print('GPU:',torch.cuda.get_device_name(0),flush=True)


In [ ]:
import os, sys, json
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['HF_HUB_OFFLINE']='1'
runtime=Path('/kaggle/working/runtime'); (runtime/'v16').mkdir(parents=True,exist_ok=True)
(runtime/'v16'/'__init__.py').write_text('"""V16: dynamic MRI training and inference, with explicit data provenance."""\n', encoding='utf-8')
(runtime/'v16'/'__main__.py').write_text("import argparse\nimport json\n\nfrom .common import config\nfrom .runner import prepare, train, merge, infer\nfrom .evaluation import evaluate\n\n\ndef main():\n    parser = argparse.ArgumentParser(description='V16 dynamic MRI pipeline')\n    sub = parser.add_subparsers(dest='command', required=True)\n    prep = sub.add_parser('prepare', help='Audit groups, freeze folds and cache real DICOMs')\n    prep.add_argument('--data-root', required=True)\n    prep.add_argument('--work', required=True)\n    prep.add_argument('--config', required=True)\n    prep.add_argument('--report-labels')\n    prep.add_argument('--groups')\n    fit = sub.add_parser('train', help='Train one fold; resumes exact matching checkpoint')\n    fit.add_argument('--work', required=True)\n    fit.add_argument('--fold', type=int, required=True)\n    fit.add_argument('--seed', type=int)\n    fit.add_argument('--arm', choices=['none', 'raw', 'platt'])\n    combine = sub.add_parser('merge', help='Verify every fold and assemble complete OOF')\n    combine.add_argument('--work', required=True)\n    combine.add_argument('--seed', type=int, required=True)\n    combine.add_argument('--arm', choices=['none', 'raw', 'platt'], required=True)\n    predict = sub.add_parser('infer', help='Offline prediction using trained V16 checkpoints')\n    predict.add_argument('--data-root', required=True)\n    predict.add_argument('--checkpoints', nargs='+', required=True)\n    predict.add_argument('--output', default='submission.csv')\n    predict.add_argument('--cache', required=True)\n    predict.add_argument('--device', choices=['auto', 'cpu', 'cuda'], default='auto')\n    predict.add_argument('--encoder-code', help='Attached local DINOv2 repo, for native Meta checkpoints')\n    predict.add_argument('--baseline-csv')\n    predict.add_argument('--blend-weight', type=float)\n    predict.add_argument('--max-seconds', type=float, default=28800)\n    score = sub.add_parser('evaluate', help='Paired expert-label group-bootstrap comparison')\n    score.add_argument('--work', required=True)\n    score.add_argument('--baseline-oof', required=True)\n    score.add_argument('--candidate-oof', required=True)\n    score.add_argument('--output', required=True)\n    score.add_argument('--weight', type=float, default=1.)\n    score.add_argument('--bootstrap', type=int, default=2000)\n    args = parser.parse_args()\n    if args.command == 'prepare':\n        result = prepare(args.data_root, args.work, config(args.config), args.report_labels, args.groups)\n    elif args.command == 'train':\n        result = train(args.work, args.fold, args.seed, args.arm)\n    elif args.command == 'merge':\n        result = merge(args.work, args.seed, args.arm)\n    elif args.command == 'evaluate':\n        result = evaluate(args.work, args.baseline_oof, args.candidate_oof, args.output,\n                          args.weight, args.bootstrap)\n    else:\n        result = infer(args.data_root, args.checkpoints, args.output, args.cache, args.device,\n                       args.baseline_csv, args.blend_weight, args.encoder_code, args.max_seconds)\n    print(json.dumps(result, indent=2, allow_nan=False))\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
(runtime/'v16'/'common.py').write_text("import hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open('rb') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b''):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + '.tmp')\n    temp.write_text(json.dumps(value, indent=2, allow_nan=False) + '\\n', encoding='utf-8')\n    os.replace(temp, path)\n\n\ndef atomic_csv(path, frame):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + '.tmp')\n    frame.to_csv(temp, index=False)\n    os.replace(temp, path)\n\n\ndef config(path):\n    defaults = json.loads(Path(__file__).with_name('config.json').read_text())\n    supplied = json.loads(Path(path).read_text())\n    if set(supplied) - set(defaults):\n        raise ValueError(f'Unknown configuration keys: {set(supplied) - set(defaults)}')\n    defaults.update(supplied)\n    for key in ('image_size', 'max_centres', 'max_series', 'epochs', 'batch_size', 'encoder_chunk', 'hidden'):\n        if not isinstance(defaults[key], int) or defaults[key] < 1:\n            raise ValueError(f'{key} must be a positive integer')\n    if not defaults['offsets'] or not all(isinstance(x, int) for x in defaults['offsets']):\n        raise ValueError('offsets must be nonempty integers')\n    if defaults['encoder'] not in ('dinov2', 'cnn', 'tiny'):\n        raise ValueError('encoder must be dinov2, cnn, or tiny (tests only)')\n    if defaults['auxiliary'] not in ('none', 'raw', 'platt'):\n        raise ValueError('auxiliary must be none, raw, or platt')\n    if defaults['expert_loss'] not in ('bce', 'asl'):\n        raise ValueError('expert_loss must be bce or asl')\n    if not 0 <= defaults['series_dropout'] < 1 or not 0 < defaults['expert_fraction'] < 1:\n        raise ValueError('Invalid sampling/dropout fractions')\n    if not 0 <= defaults['aux_cap'] <= 1 or defaults['folds'] < 2 or defaults['policy_bootstrap'] < 20:\n        raise ValueError('Invalid auxiliary cap, fold count, or bootstrap count')\n    lo, hi = defaults['percentiles']\n    if not 0 <= lo < hi <= 100 or defaults['crop_mm'] <= 0:\n        raise ValueError('Invalid intensity/crop configuration')\n    return defaults\n\n\ndef schema(root, id_col):\n    columns = pd.read_csv(Path(root) / 'sample_submission.csv', nrows=0).columns.tolist()\n    if id_col not in columns:\n        raise ValueError(f'Submission schema lacks {id_col}')\n    targets = [c for c in columns if c != id_col]\n    if not targets:\n        raise ValueError('No finding columns')\n    return {'id_col': id_col, 'targets': targets}\n\n\ndef read_studies(root, split, contract):\n    frame = pd.read_csv(Path(root) / f'{split}.csv', dtype={contract['id_col']: str})\n    check_ids(frame, contract['id_col'])\n    return frame\n\n\ndef check_ids(frame, id_col):\n    if id_col not in frame or frame[id_col].isna().any():\n        raise ValueError('Missing study IDs')\n    ids = frame[id_col].astype(str)\n    if ids.duplicated().any() or ids.str.strip().eq('').any():\n        raise ValueError('Duplicate or empty study IDs')\n    return ids.tolist()\n\n\ndef child(root, uid):\n    root = Path(root).resolve()\n    if not uid or '/' in uid or '\\\\' in uid or uid in ('.', '..'):\n        raise ValueError('Invalid UID path component')\n    path = (root / uid).resolve()\n    if not path.is_relative_to(root):\n        raise ValueError('UID path escaped root')\n    return path\n\n\ndef aligned(frame, ids, contract, probabilities=False):\n    id_col, targets = contract['id_col'], contract['targets']\n    actual = check_ids(frame, id_col)\n    if set(actual) != set(ids):\n        raise ValueError('Study coverage differs from the expected cohort')\n    frame = frame.assign(**{id_col: frame[id_col].astype(str)})\n    values = frame.set_index(id_col).loc[ids, targets].to_numpy(float)\n    valid = np.isfinite(values) & (values >= 0) & (values <= 1)\n    if not (valid if probabilities else (valid | np.isnan(values))).all():\n        raise ValueError('Invalid probabilities/labels')\n    return values\n\n\ndef write_predictions(path, ids, values, contract):\n    if values.shape != (len(ids), len(contract['targets'])):\n        raise ValueError('Prediction shape differs from schema')\n    frame = pd.DataFrame(values, columns=contract['targets'])\n    frame.insert(0, contract['id_col'], ids)\n    aligned(frame, ids, contract, probabilities=True)\n    atomic_csv(path, frame)\n    return frame\n", encoding='utf-8')
(runtime/'v16'/'data.py').write_text('"""Real single-frame MRI DICOM decoding and versioned, per-study disk caches."""\nimport json\nimport os\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\nfrom torch.nn.utils.rnn import pad_sequence\nfrom torch.utils.data import Dataset\n\nfrom .common import atomic_json, child, fingerprint, sha\n\nDECODER_VERSION = \'v16-dicom-2-compact\'\nTRANSFORM_KEYS = (\'image_size\', \'crop_mm\', \'offsets\', \'max_centres\', \'max_series\', \'percentiles\')\n\n\ndef transform_contract(cfg):\n    import pydicom\n    return {\'version\': DECODER_VERSION, \'pydicom\': pydicom.__version__,\n            \'torch\': str(torch.__version__), \'numpy\': np.__version__,\n            \'cache_dtype\': \'float16-normalized\',\n            **{k: cfg[k] for k in TRANSFORM_KEYS}}\n\n\ndef _vector(ds, key, n):\n    try:\n        value = np.asarray(getattr(ds, key), float)\n        return value if value.shape == (n,) and np.isfinite(value).all() else None\n    except (AttributeError, ValueError, TypeError):\n        return None\n\n\ndef _number(value):\n    try:\n        value = float(value)\n        return value if np.isfinite(value) else None\n    except (ValueError, TypeError):\n        return None\n\n\ndef _flag(value):\n    if pd.isna(value):\n        return -1.\n    text = str(value).strip().lower()\n    return {\'1\': 1., \'1.0\': 1., \'true\': 1., \'0\': 0., \'0.0\': 0., \'false\': 0.}.get(text, -1.)\n\n\ndef resize_pad(array, size):\n    x = torch.as_tensor(np.ascontiguousarray(array), dtype=torch.float32)\n    h, w = x.shape\n    scale = min(size / h, size / w)\n    rh, rw = max(1, round(h * scale)), max(1, round(w * scale))\n    x = F.interpolate(x[None, None], size=(rh, rw), mode=\'bilinear\',\n                      align_corners=False, antialias=True)[0, 0]\n    top, left = (size - rh) // 2, (size - rw) // 2\n    return F.pad(x, (left, size-rw-left, top, size-rh-top))\n\n\ndef decode_series(directory, metadata, cfg, expected_study=None, expected_series=None):\n    import pydicom\n    from pydicom.pixels import apply_modality_lut\n    events, records, seen = [], [], set()\n    for path in sorted(p for p in Path(directory).rglob(\'*\') if p.is_file() and p.suffix.lower() == \'.dcm\'):\n        try:\n            ds = pydicom.dcmread(path, stop_before_pixels=True)\n            if expected_study is not None and str(getattr(ds, \'StudyInstanceUID\', \'\')) != expected_study:\n                raise ValueError(\'DICOM study identity differs from containing study\')\n            if expected_series is not None and str(getattr(ds, \'SeriesInstanceUID\', \'\')) != expected_series:\n                raise ValueError(\'DICOM series identity differs from containing series\')\n            sop = str(getattr(ds, \'SOPInstanceUID\', path))\n            if sop in seen:\n                events.append({\'kind\': \'duplicate_sop\', \'file\': str(path)})\n                continue\n            seen.add(sop)\n            if int(getattr(ds, \'NumberOfFrames\', 1)) != 1:\n                raise ValueError(\'Enhanced/multiframe DICOM requires a separate adapter\')\n            iop, ipp = _vector(ds, \'ImageOrientationPatient\', 6), _vector(ds, \'ImagePositionPatient\', 3)\n            records.append({\'path\': path, \'ds\': ds, \'iop\': iop, \'ipp\': ipp,\n                            \'spacing\': _vector(ds, \'PixelSpacing\', 2),\n                            \'instance\': _number(getattr(ds, \'InstanceNumber\', None))})\n        except Exception as exc:\n            events.append({\'kind\': \'header_failure\', \'file\': str(path), \'error\': str(exc)})\n    if not records:\n        return None, events + [{\'kind\': \'no_readable_headers\'}]\n    # Separate orientations/acquisitions rather than forming a false sequence.\n    stacks = {}\n    for r in records:\n        orientation = tuple(np.round(r[\'iop\'], 3)) if r[\'iop\'] is not None else None\n        key = (str(getattr(r[\'ds\'], \'SeriesInstanceUID\', \'\')),\n               str(getattr(r[\'ds\'], \'AcquisitionNumber\', \'\')),\n               str(getattr(r[\'ds\'], \'EchoNumbers\', \'\')), orientation)\n        stacks.setdefault(key, []).append(r)\n    records = max(stacks.values(), key=len)  # deterministic sorted-path tie order\n    if len(stacks) > 1:\n        events.append({\'kind\': \'selected_largest_consistent_stack\', \'stacks\': len(stacks)})\n    iop = records[0][\'iop\']\n    normal = None if iop is None else np.cross(iop[:3], iop[3:])\n    geometry = normal is not None and np.linalg.norm(normal) > .99 and all(r[\'ipp\'] is not None for r in records)\n    if geometry:\n        normal = normal / np.linalg.norm(normal)\n        if normal[np.argmax(np.abs(normal))] < 0:\n            normal = -normal\n        records.sort(key=lambda r: (float(r[\'ipp\'] @ normal), str(r[\'path\'])))\n        unique = {}\n        for r in records:\n            unique.setdefault(round(float(r[\'ipp\'] @ normal), 4), r)\n        if len(unique) != len(records):\n            events.append({\'kind\': \'duplicate_position\', \'removed\': len(records) - len(unique)})\n        records = list(unique.values())\n    elif all(r[\'instance\'] is not None for r in records):\n        records.sort(key=lambda r: (r[\'instance\'], str(r[\'path\'])))\n        events.append({\'kind\': \'instance_order_fallback\'})\n    else:\n        return None, events + [{\'kind\': \'unorderable_stack\'}]\n    spacing = [r[\'spacing\'] for r in records if r[\'spacing\'] is not None and (r[\'spacing\'] > 0).all()]\n    shared_spacing = np.median(spacing, axis=0) if spacing else None\n    count = min(len(records), cfg[\'max_centres\'])\n    centres = np.linspace(0, len(records)-1, count).round().astype(int)\n    neighbors = np.clip(centres[:, None] + np.asarray(cfg[\'offsets\'])[None], 0, len(records)-1)\n    required = np.unique(neighbors)\n    decoded, pixels = {}, []\n    spacing_unknown = False\n    for index in required:\n        r = records[index]\n        try:\n            ds = pydicom.dcmread(r[\'path\'])\n            raw = ds.pixel_array\n            if raw.ndim != 2 or not np.isfinite(raw).all():\n                raise ValueError(\'Expected finite single-frame monochrome pixels\')\n            a = apply_modality_lut(raw, ds).astype(np.float32)\n            valid = np.isfinite(a)\n            padding = getattr(ds, \'PixelPaddingValue\', None)\n            if padding is not None:\n                valid &= raw != padding\n            if not valid.any():\n                raise ValueError(\'All pixels missing/padding\')\n            if getattr(ds, \'PhotometricInterpretation\', \'\') == \'MONOCHROME1\':\n                a = a[valid].max() + a[valid].min() - a\n            ps = r[\'spacing\']\n            if ps is None or not (ps > 0).all():\n                ps = shared_spacing\n                events.append({\'kind\': \'spacing_fallback\', \'file\': str(r[\'path\']),\n                               \'policy\': \'series_median\' if ps is not None else \'full_fov\'})\n            if ps is not None:\n                h, w = a.shape\n                ch, cw = min(h, max(1, round(cfg[\'crop_mm\']/ps[0]))), min(w, max(1, round(cfg[\'crop_mm\']/ps[1])))\n                y, x = (h-ch)//2, (w-cw)//2\n                a, valid = a[y:y+ch, x:x+cw], valid[y:y+ch, x:x+cw]\n            else:\n                spacing_unknown = True\n            if not valid.any():\n                raise ValueError(\'Crop contains only padding\')\n            if iop is not None:\n                if iop[:3][np.argmax(np.abs(iop[:3]))] < 0:\n                    a, valid = a[:, ::-1], valid[:, ::-1]\n                if iop[3:][np.argmax(np.abs(iop[3:]))] < 0:\n                    a, valid = a[::-1], valid[::-1]\n            # Deterministic bounded intensity sample; shared across this series.\n            vals = a[valid].ravel()\n            pixels.append(vals[::max(1, len(vals)//65536)])\n            decoded[int(index)] = (a, valid)\n        except Exception as exc:\n            events.append({\'kind\': \'pixel_failure\', \'file\': str(r[\'path\']), \'error\': str(exc)})\n    if not decoded:\n        return None, events + [{\'kind\': \'no_decodable_pixels\'}]\n    lo, hi = np.percentile(np.concatenate(pixels), cfg[\'percentiles\'])\n    if not hi > lo:\n        return None, events + [{\'kind\': \'constant_series\'}]\n    tiles = {}\n    for index, (a, valid) in decoded.items():\n        a = np.where(valid, np.clip((a-lo)/(hi-lo), 0, 1), 0)\n        tiles[index] = resize_pad(a, cfg[\'image_size\'])\n    windows, positions, selected_neighbors = [], [], []\n    for center, indices in zip(centres, neighbors):\n        if not all(int(i) in tiles for i in indices):\n            events.append({\'kind\': \'window_missing_neighbor\', \'center\': int(center)})\n            continue\n        windows.append(torch.stack([tiles[int(i)] for i in indices]))\n        selected_neighbors.append([int(i) for i in indices])\n        if geometry:\n            positions.append(float(records[int(center)][\'ipp\'] @ normal))\n        else:\n            positions.append(float(center))\n    if not windows:\n        return None, events + [{\'kind\': \'no_complete_windows\'}]\n    positions = np.asarray(positions, np.float32)\n    positions = (positions-positions.min()) / max(float(np.ptp(positions)), 1e-6)\n    plane = np.zeros(3, np.float32)\n    if normal is not None:\n        plane[np.argmax(np.abs(normal))] = 1\n    meta = np.r_[plane, _flag(metadata.get(\'Fluid_Sensitive\', np.nan)),\n                 _flag(metadata.get(\'Fat_Suppression\', np.nan)), float(spacing_unknown or not geometry)]\n    used = sorted({i for window in selected_neighbors for i in window})\n    mapping = {index: i for i, index in enumerate(used)}\n    return {\'x\': torch.stack(windows).half(), \'position\': torch.from_numpy(positions),\n            \'tiles\': torch.stack([tiles[i] for i in used]).half(),\n            \'indices\': torch.tensor([[mapping[i] for i in window] for window in selected_neighbors]),\n            \'meta\': torch.tensor(meta, dtype=torch.float32)}, events\n\n\ndef series_for_study(directory, uid, metadata, cfg):\n    series, events = [], []\n    dirs = sorted(p for p in directory.iterdir() if p.is_dir()) if directory.exists() else []\n    dirs.sort(key=lambda p: (-sum(f.suffix.lower() == \'.dcm\' for f in p.rglob(\'*\') if f.is_file()), p.name))\n    for sdir in dirs:\n        decoded, reasons = decode_series(sdir, metadata.get((uid, sdir.name), {}), cfg,\n                                         expected_study=uid, expected_series=sdir.name)\n        events.extend({\'series\': sdir.name, **r} for r in reasons)\n        if decoded is not None:\n            series.append(decoded)\n        if len(series) >= cfg[\'max_series\']:\n            break\n    return series, events\n\n\ndef build_cache(root, split, ids, cfg, cache_dir):\n    root, cache_dir = Path(root), Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    contract = transform_contract(cfg)\n    meta_path = root / f\'{split}_series.csv\'\n    metadata = {}\n    if meta_path.exists():\n        table = pd.read_csv(meta_path, dtype={cfg[\'id_col\']: str, cfg[\'series_col\']: str})\n        if table[[cfg[\'id_col\'], cfg[\'series_col\']]].isna().any().any() or table.duplicated([cfg[\'id_col\'], cfg[\'series_col\']]).any():\n            raise ValueError(\'Missing/duplicate study-series metadata\')\n        metadata = {(r[cfg[\'id_col\']], r[cfg[\'series_col\']]): r for r in table.to_dict(\'records\')}\n    entries = {}\n    for number, uid in enumerate(ids):\n        directory = child(root / f\'{split}_series\', uid)\n        files = sorted(p for p in directory.rglob(\'*\') if p.is_file()) if directory.exists() else []\n        # Content fingerprints allow safe reuse across mount paths and detect changed pixels.\n        source = [(str(p.relative_to(directory)), sha(p)) for p in files]\n        series_meta = {k[1]: {str(c): (None if pd.isna(v) else v) for c, v in r.items()}\n                       for k, r in metadata.items() if k[0] == uid}\n        key = fingerprint({\'uid\': uid, \'source\': source, \'metadata\': series_meta, \'contract\': contract})\n        target = cache_dir / f\'{key}.pt\'\n        log = cache_dir / f\'{key}.json\'\n        if not target.exists() or not log.exists() or sha(target) != json.loads(log.read_text())[\'sha256\']:\n            series, events = series_for_study(directory, uid, metadata, cfg)\n            # Save each normalized slice once; reconstruct overlapping triplets at load time.\n            series = [{k: v for k, v in s.items() if k != \'x\'} for s in series]\n            estimate = sum(v.numel()*v.element_size() for s in series for v in s.values())\n            if shutil.disk_usage(cache_dir).free < estimate + 1024**3:\n                raise RuntimeError(\'Insufficient cache disk space: use larger local scratch or a smaller new preprocessing configuration\')\n            temp = target.with_suffix(\'.tmp\')\n            torch.save({\'series\': series}, temp)\n            os.replace(temp, target)\n            atomic_json(log, {\'uid\': uid, \'sha256\': sha(target), \'series\': len(series), \'events\': events})\n        info = json.loads(log.read_text())\n        entries[uid] = {\'file\': target.name, **info}\n        if (number+1) % 25 == 0 or number+1 == len(ids):\n            print(f\'Cached {number+1}/{len(ids)} studies\', flush=True)\n    manifest = {\'split\': split, \'contract\': contract, \'entries\': entries,\n                \'missing_studies\': [u for u, e in entries.items() if e[\'series\'] == 0]}\n    atomic_json(cache_dir / \'manifest.json\', manifest)\n    return manifest\n\n\nclass CachedStudies(Dataset):\n    def __init__(self, cache_dir, ids, training=False, dropout=0.):\n        self.root, self.ids = Path(cache_dir), list(ids)\n        self.manifest = json.loads((self.root / \'manifest.json\').read_text())\n        if not set(ids).issubset(self.manifest[\'entries\']):\n            raise ValueError(\'Cache missing expected IDs\')\n        self.training, self.dropout = training, dropout\n        # Verify once when opening the dataset, rather than hash every epoch.\n        for uid in ids:\n            entry = self.manifest[\'entries\'][uid]\n            if sha(self.root / entry[\'file\']) != entry[\'sha256\']:\n                raise ValueError(f\'Cache checksum mismatch for {uid}\')\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, i):\n        uid = self.ids[i]\n        record = torch.load(self.root / self.manifest[\'entries\'][uid][\'file\'], weights_only=True)\n        series = [{**s, \'x\': s[\'tiles\'][s[\'indices\']]} for s in record[\'series\']]\n        if self.training and len(series) > 1:\n            keep = torch.rand(len(series)) >= self.dropout\n            if not keep.any():\n                keep[torch.randint(len(series), (1,))] = True\n            series = [s for s, k in zip(series, keep) if k]\n        if self.training:\n            # Shared small intensity transform; no anatomy-changing random flips.\n            gain = .95 + .1 * torch.rand(1).item()\n            series = [{**s, \'x\': (s[\'x\'].float()*gain).clamp(0, 1)} for s in series]\n        return {\'uid\': uid, \'series\': series}\n\n\nclass DicomStudies(Dataset):\n    """Streaming hidden-set inference: never caches the entire test cohort."""\n    def __init__(self, root, ids, cfg):\n        self.root, self.ids, self.cfg = Path(root), list(ids), cfg\n        self.metadata, self.events = {}, {}\n        path = self.root/\'test_series.csv\'\n        if path.exists():\n            table = pd.read_csv(path, dtype={cfg[\'id_col\']: str, cfg[\'series_col\']: str})\n            if table[[cfg[\'id_col\'], cfg[\'series_col\']]].isna().any().any() or table.duplicated([cfg[\'id_col\'], cfg[\'series_col\']]).any():\n                raise ValueError(\'Missing/duplicate test series metadata\')\n            self.metadata = {(r[cfg[\'id_col\']], r[cfg[\'series_col\']]): r for r in table.to_dict(\'records\')}\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, i):\n        uid = self.ids[i]\n        series, events = series_for_study(child(self.root/\'test_series\', uid), uid, self.metadata, self.cfg)\n        self.events[uid] = events\n        return {\'uid\': uid, \'series\': [{k: v for k, v in s.items() if k not in (\'tiles\', \'indices\')} for s in series]}\n\n\ndef collate(items):\n    series, owners = [], []\n    for i, item in enumerate(items):\n        series.extend(item[\'series\'])\n        owners.extend([i]*len(item[\'series\']))\n    return {\'ids\': [x[\'uid\'] for x in items],\n            \'x\': pad_sequence([s[\'x\'].float() for s in series], batch_first=True) if series else None,\n            \'lengths\': torch.tensor([len(s[\'x\']) for s in series]),\n            \'position\': pad_sequence([s[\'position\'] for s in series], batch_first=True) if series else None,\n            \'metadata\': torch.stack([s[\'meta\'] for s in series]) if series else None,\n            \'owners\': torch.tensor(owners, dtype=torch.long)}\n\n\ndef to_device(batch, device):\n    return {k: v.to(device) if torch.is_tensor(v) and k != \'lengths\' else v for k, v in batch.items()}\n', encoding='utf-8')
(runtime/'v16'/'model.py').write_text('"""Offline pretrained encoders with within-series GRU and target attention."""\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence\n\n\nclass Encoder(nn.Module):\n    def __init__(self, cfg, initialize=True, spec=None):\n        super().__init__()\n        self.kind = cfg[\'encoder\']\n        self.spec = spec or {}\n        channels = len(cfg[\'offsets\'])\n        if self.kind == \'tiny\':\n            self.net = nn.Sequential(nn.Conv2d(channels, 8, 3, padding=1), nn.GELU(),\n                                     nn.AdaptiveAvgPool2d(1), nn.Flatten())\n            self.out_dim, self.spec = 8, {\'api\': \'tiny\'}\n            return\n        if channels != 3:\n            raise ValueError(\'These pretrained backbones require three configured context channels\')\n        self.register_buffer(\'mean\', torch.tensor([.485, .456, .406])[None, :, None, None])\n        self.register_buffer(\'std\', torch.tensor([.229, .224, .225])[None, :, None, None])\n        path = Path(cfg[\'encoder_path\'])\n        if self.kind == \'dinov2\':\n            api = self.spec.get(\'api\') or (\'hf\' if (path / \'config.json\').is_file() else \'meta\')\n            if api == \'hf\':\n                from transformers import Dinov2Config, Dinov2Model\n                if initialize:\n                    self.net = Dinov2Model.from_pretrained(str(path), local_files_only=True)\n                else:\n                    self.net = Dinov2Model(Dinov2Config.from_dict(self.spec[\'config\']))\n                blocks, norm = self.net.encoder.layer, self.net.layernorm\n                self.out_dim = self.net.config.hidden_size\n                self.spec = {\'api\': \'hf\', \'config\': self.net.config.to_dict()}\n                patch = self.net.config.patch_size\n            else:\n                repo = Path(cfg[\'dinov2_repo\'])\n                if not (repo / \'hubconf.py\').is_file():\n                    raise ValueError(\'Native Meta .pth needs dinov2_repo pointing to an attached local DINOv2 code repository\')\n                self.net = torch.hub.load(str(repo), \'dinov2_vits14\', source=\'local\', pretrained=False)\n                if initialize:\n                    if not path.is_file():\n                        raise ValueError(\'encoder_path must be the native DINOv2 Small .pth file\')\n                    state = torch.load(path, map_location=\'cpu\', weights_only=True)\n                    self.net.load_state_dict(state, strict=True)\n                blocks, norm = self.net.blocks, self.net.norm\n                self.out_dim = self.net.embed_dim\n                self.spec = {\'api\': \'meta\', \'variant\': \'dinov2_vits14\'}\n                patch = self.net.patch_size\n            if cfg[\'image_size\'] % patch:\n                raise ValueError(f\'image_size must be divisible by DINOv2 patch size {patch}\')\n            for p in self.net.parameters():\n                p.requires_grad = False\n            n = cfg[\'unfreeze_last\']\n            if n < 0 or n > len(blocks):\n                raise ValueError(\'unfreeze_last exceeds encoder depth\')\n            if n:\n                for block in blocks[-n:]:\n                    for p in block.parameters():\n                        p.requires_grad = True\n                for p in norm.parameters():\n                    p.requires_grad = True\n            # Checkpoint chunks at the wrapper boundary below, also for native Meta.\n        else:\n            import timm\n            self.net = timm.create_model(cfg[\'cnn_arch\'], pretrained=False, num_classes=0, global_pool=\'avg\')\n            if initialize:\n                if not path.is_file():\n                    raise ValueError(\'CNN training requires an attached compatible encoder state dictionary\')\n                state = torch.load(path, map_location=\'cpu\', weights_only=True)\n                # A backbone-only checkpoint is required; silent key dropping is prohibited.\n                self.net.load_state_dict(state, strict=True)\n            self.out_dim = self.net.num_features\n            self.spec = {\'api\': \'timm\', \'arch\': cfg[\'cnn_arch\']}\n\n    def forward(self, x):\n        if self.kind == \'tiny\':\n            return self.net(x)\n        x = (x-self.mean)/self.std\n        if self.spec[\'api\'] == \'hf\':\n            return self.net(pixel_values=x).last_hidden_state[:, 0]\n        if self.spec[\'api\'] == \'meta\':\n            return self.net.forward_features(x)[\'x_norm_clstoken\']\n        return self.net(x)\n\n\nclass KneeModel(nn.Module):\n    def __init__(self, cfg, n_targets, initialize=True, encoder_spec=None):\n        super().__init__()\n        self.encoder = Encoder(cfg, initialize, encoder_spec)\n        self.cfg = cfg\n        dim, hidden = self.encoder.out_dim, cfg[\'hidden\']\n        self.metadata = nn.Linear(7, dim)  # patient-axis plane(3), fluid/fat(2), missing geometry, position\n        self.gru = nn.GRU(dim, hidden, bidirectional=True, batch_first=True)\n        self.query = nn.Parameter(torch.randn(n_targets, hidden*2)*.02)\n        self.weight = nn.Parameter(torch.randn(n_targets, hidden*2)*.02)\n        self.bias = nn.Parameter(torch.zeros(n_targets))\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Preserve pretrained BN statistics with study/microbatches as small as one.\n        # ConvNeXt and DINO use layer normalization, but alternate CNNs may use BN.\n        if mode:\n            for module in self.encoder.modules():\n                if isinstance(module, nn.modules.batchnorm._BatchNorm):\n                    module.eval()\n        return self\n\n    def forward(self, batch):\n        n = len(batch[\'ids\'])\n        if batch[\'x\'] is None:\n            return self.bias.expand(n, -1), torch.zeros(n, dtype=torch.bool, device=self.bias.device)\n        x, lengths = batch[\'x\'], batch[\'lengths\']\n        valid = torch.arange(x.shape[1], device=x.device)[None] < lengths.to(x.device)[:, None]\n        chunks = []\n        for images in x[valid].split(self.cfg[\'encoder_chunk\']):\n            if self.training and self.cfg[\'gradient_checkpointing\'] and any(p.requires_grad for p in self.encoder.parameters()):\n                from torch.utils.checkpoint import checkpoint\n                chunks.append(checkpoint(self.encoder, images, use_reentrant=False))\n            else:\n                chunks.append(self.encoder(images))\n        features = torch.cat(chunks)\n        # T4 FP16 has limited exponent range. Keep recurrence and finding\n        # attention in FP32 while retaining mixed precision for the encoder.\n        with torch.autocast(x.device.type, enabled=False):\n            return self._head(batch, features.float(), valid, lengths, n)\n\n    def _head(self, batch, features, valid, lengths, n):\n        padded = features.new_zeros(*valid.shape, features.shape[-1])\n        padded[valid] = features\n        meta = torch.cat([batch[\'metadata\'][:, None].expand(-1, valid.shape[1], -1),\n                          batch[\'position\'][..., None]], dim=-1)\n        padded = padded + self.metadata(meta).to(padded.dtype)\n        packed = pack_padded_sequence(padded, lengths.cpu(), batch_first=True, enforce_sorted=False)\n        contextual, _ = self.gru(packed)\n        contextual, _ = pad_packed_sequence(contextual, batch_first=True)\n        outputs, available = [], []\n        for owner in range(n):\n            tokens = contextual[(batch[\'owners\'] == owner)[:, None] & valid]\n            if not len(tokens):\n                outputs.append(self.bias)\n                available.append(False)\n            else:\n                att = (self.query @ tokens.T / tokens.shape[-1]**.5).softmax(-1)\n                outputs.append(((att @ tokens)*self.weight).sum(-1)+self.bias)\n                available.append(True)\n        return torch.stack(outputs), torch.tensor(available, device=features.device)\n\n\ndef masked_loss(logits, y, mask, cfg, auxiliary=False, weights=None):\n    z = logits.float()\n    mask = mask.bool() & torch.isfinite(y)\n    safe = torch.where(mask, y.float(), torch.zeros_like(z))\n    loss = F.binary_cross_entropy_with_logits(z, safe, reduction=\'none\')\n    if not auxiliary and cfg[\'expert_loss\'] == \'asl\':\n        p = z.sigmoid()\n        loss = -safe*(1-p).pow(cfg[\'gamma_pos\'])*F.logsigmoid(z)\n        loss -= (1-safe)*p.pow(cfg[\'gamma_neg\'])*F.logsigmoid(-z)\n    if weights is not None:\n        loss = loss * weights\n    return (loss*mask).sum()/mask.sum().clamp_min(1)\n', encoding='utf-8')
(runtime/'v16'/'validation.py').write_text('import numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\nfrom sklearn.linear_model import LogisticRegression\n\nfrom .common import check_ids, fingerprint\n\n\ndef auc(y, p):\n    observed = np.isfinite(y)\n    y, p = y[observed], p[observed]\n    pos, neg = int((y == 1).sum()), int((y == 0).sum())\n    if not pos or not neg:\n        return None\n    return float((rankdata(p)[y == 1].sum()-pos*(pos+1)/2)/(pos*neg))\n\n\ndef metrics(y, p, targets):\n    if y.shape != p.shape or not np.isfinite(p).all():\n        raise ValueError(\'Invalid metric matrix\')\n    values = [auc(y[:, j], p[:, j]) for j in range(y.shape[1])]\n    return {\'macro_auc\': float(np.mean(values)) if all(v is not None for v in values) else None,\n            \'targets\': {t: {\'auc\': values[j], \'positive\': int((y[:, j] == 1).sum()),\n                             \'negative\': int((y[:, j] == 0).sum()), \'unknown\': int(np.isnan(y[:, j]).sum())}\n                        for j, t in enumerate(targets)},\n            \'all_targets_scorable\': all(v is not None for v in values)}\n\n\ndef load_groups(path, ids, id_col):\n    if path is None:\n        return np.asarray(ids), \'Study-only grouping; patient separation is unverified\'\n    frame = pd.read_csv(path, dtype=str)\n    actual = check_ids(frame, id_col)\n    if set(actual) != set(ids) or \'group_id\' not in frame or frame.group_id.isna().any() or frame.group_id.str.strip().eq(\'\').any():\n        raise ValueError(\'Groups CSV must cover all studies with nonempty group_id\')\n    return frame.set_index(id_col).loc[ids, \'group_id\'].to_numpy(), \'User-supplied patient/duplicate grouping; verify identity provenance\'\n\n\ndef make_folds(ids, y, groups, k, seed):\n    """Group-preserving greedy multilabel count allocation, with support audits."""\n    if not (np.isnan(y) | (y == 0) | (y == 1)).all():\n        raise ValueError(\'Expert targets must be binary or missing\')\n    groups = np.asarray(groups)\n    names, inverse = np.unique(groups, return_inverse=True)\n    if len(names) < k:\n        raise ValueError(\'Fewer independent groups than folds\')\n    features = np.c_[y == 1, y == 0, np.isfinite(y).any(1), np.ones(len(y))].astype(float)\n    counts = np.stack([features[inverse == i].sum(0) for i in range(len(names))])\n    totals = counts.sum(0)\n    scale = np.maximum(totals/k, 1)\n    active = totals > 0\n    rng = np.random.default_rng(seed)\n    ties = rng.random(len(names))\n    rarity = ((counts[:, :y.shape[1]] > 0)/np.maximum(totals[:y.shape[1]], 1)).sum(1)\n    order = sorted(range(len(names)), key=lambda i: (-rarity[i], -counts[i, -1], ties[i]))\n    allocation = np.zeros((k, counts.shape[1]))\n    assignment = np.full(len(names), -1)\n    tie_order = rng.permutation(k).tolist()\n    for step, g in enumerate(order):\n        candidates = [tie_order[step]] if step < k else tie_order\n        def cost(f):\n            before = ((allocation[f]-totals/k)/scale)**2\n            after = ((allocation[f]+counts[g]-totals/k)/scale)**2\n            return (after-before)[active].sum()\n        f = min(candidates, key=cost)\n        allocation[f] += counts[g]\n        assignment[g] = f\n    fold = assignment[inverse]\n    audit = []\n    for f in range(k):\n        tr, va = fold != f, fold == f\n        assert not set(groups[tr]) & set(groups[va])\n        audit.append({\'fold\': f, \'train_studies\': int(tr.sum()), \'val_studies\': int(va.sum()),\n                      \'positive\': (y[va] == 1).sum(0).tolist(), \'negative\': (y[va] == 0).sum(0).tolist(),\n                      \'train_scorable\': ((y[tr] == 1).any(0) & (y[tr] == 0).any(0)).tolist(),\n                      \'val_scorable\': ((y[va] == 1).any(0) & (y[va] == 0).any(0)).tolist()})\n    return fold, {\'method\': \'group-preserving multilabel greedy allocation; approximate balance\',\n                  \'k\': k, \'seed\': seed, \'audit\': audit,\n                  \'assignment_hash\': fingerprint(dict(zip(ids, fold.tolist())))}\n\n\ndef supervision(y, report, groups, train_idx, cfg, targets, ids):\n    """Calibrators and raw-source transfer gate see training-fold expert cells ONLY."""\n    expert_mask = np.isfinite(y)\n    addressed = np.isfinite(report) & (report != cfg[\'silent_value\'])\n    aux_mask = addressed & ~expert_mask\n    weights = np.zeros(y.shape[1], np.float32)\n    calibrated = report.copy()\n    policies = {}\n    train_mask = np.zeros(len(y), bool)\n    train_mask[train_idx] = True\n    for j, target in enumerate(targets):\n        selected = np.flatnonzero(train_mask & expert_mask[:, j] & addressed[:, j])\n        truth, raw = y[selected, j], report[selected, j]\n        policy = {\'fit_ids\': [ids[i] for i in selected], \'weight\': 0., \'calibrator\': \'identity\',\n                  \'reason\': \'disabled\' if cfg[\'auxiliary\'] == \'none\' else \'insufficient_support\'}\n        policies[target] = policy\n        if cfg[\'auxiliary\'] == \'none\' or min((truth == 1).sum(), (truth == 0).sum()) < 3:\n            continue\n        unique = np.unique(groups[selected])\n        by_group = [np.flatnonzero(groups[selected] == g) for g in unique]\n        rng = np.random.default_rng(cfg[\'fold_seed\'] + j)\n        draws = []\n        for _ in range(cfg[\'policy_bootstrap\']):\n            sample = np.concatenate([by_group[g] for g in rng.integers(len(unique), size=len(unique))])\n            value = auc(truth[sample], raw[sample])\n            if value is not None:\n                draws.append(value)\n        if len(draws) < .8 * cfg[\'policy_bootstrap\']:\n            policy[\'reason\'] = \'unreliable_group_bootstrap\'\n            continue\n        lo = float(np.quantile(draws, .025))\n        prevalence = truth.mean()\n        skill = 1 - float(np.mean((raw-truth)**2)) / float(prevalence*(1-prevalence))\n        policy.update({\'auc_lower95\': lo, \'raw_brier_skill\': skill, \'fit_group_count\': len(unique)})\n        if lo > .5 and skill > 0:\n            weight = min(skill, cfg[\'aux_cap\'])\n            support = min(len(np.unique(groups[selected][truth == 1])), len(np.unique(groups[selected][truth == 0])))\n            if support < 10:\n                weight *= .5\n            weights[j] = weight\n            policy.update({\'weight\': float(weight), \'reason\': \'raw_source_gate_passed\'})\n        else:\n            policy[\'reason\'] = \'raw_source_gate_failed\'\n        if cfg[\'auxiliary\'] == \'platt\':\n            z = np.log(np.clip(raw, 1e-4, 1-1e-4)/(1-np.clip(raw, 1e-4, 1-1e-4)))\n            try:\n                model = LogisticRegression(C=1., solver=\'lbfgs\', max_iter=1000).fit(z[:, None], truth)\n                indices = np.flatnonzero(addressed[:, j])\n                p = np.clip(report[indices, j], 1e-4, 1-1e-4)\n                calibrated[indices, j] = model.predict_proba(np.log(p/(1-p))[:, None])[:, 1]\n                policy.update({\'calibrator\': \'platt\', \'coef\': float(model.coef_[0, 0]),\n                               \'intercept\': float(model.intercept_[0])})\n            except (ValueError, FloatingPointError) as exc:\n                policy[\'calibration_error\'] = str(exc)\n    aux_mask &= weights[None] > 0\n    # Do not turn the raw-policy gate into an in-sample calibrated gate.\n    return calibrated.astype(np.float32), aux_mask, weights, policies\n', encoding='utf-8')
(runtime/'v16'/'runner.py').write_text('import json\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Subset\n\nfrom .common import (aligned, atomic_csv, atomic_json, check_ids, fingerprint,\n                     read_studies, schema, sha, write_predictions)\nfrom .data import CachedStudies, DicomStudies, build_cache, collate, to_device, transform_contract\nfrom .model import KneeModel, masked_loss\nfrom .validation import load_groups, make_folds, metrics, supervision\n\n\ndef code_hash():\n    names = (\'__init__.py\', \'__main__.py\', \'common.py\', \'data.py\', \'model.py\',\n             \'validation.py\', \'runner.py\', \'evaluation.py\')\n    return fingerprint({name: (Path(__file__).parent/name).read_text(encoding=\'utf-8\') for name in names})\n\n\ndef device_for(request):\n    if request not in (\'cpu\', \'cuda\', \'auto\'):\n        raise ValueError(\'device must be cpu, cuda, or auto\')\n    if request == \'cpu\':\n        return torch.device(\'cpu\')\n    try:\n        probe = (torch.ones(1, device=\'cuda\')*2).cpu().item()\n        if probe != 2:\n            raise RuntimeError(\'CUDA arithmetic failed\')\n        return torch.device(\'cuda\')\n    except Exception as exc:\n        if request == \'cuda\':\n            raise RuntimeError(\'Required CUDA device failed its arithmetic probe\') from exc\n        return torch.device(\'cpu\')\n\n\ndef prepare(root, work, cfg, report_path=None, groups_path=None):\n    root, work = Path(root), Path(work)\n    work.mkdir(parents=True, exist_ok=True)\n    if cfg[\'encoder\'] != \'tiny\':\n        assets = Path(cfg[\'encoder_path\'])\n        if not cfg[\'encoder_path\'] or not assets.exists():\n            raise ValueError(\'Set encoder_path to your attached pretrained encoder before preparing the cache\')\n        if cfg[\'encoder\'] == \'dinov2\' and not (assets/\'config.json\').is_file():\n            if not assets.is_file() or not (Path(cfg[\'dinov2_repo\'])/\'hubconf.py\').is_file():\n                raise ValueError(\'Native DINOv2 .pth requires dinov2_repo with local hubconf.py\')\n    contract = schema(root, cfg[\'id_col\'])\n    frame = read_studies(root, \'train\', contract)\n    ids = check_ids(frame, contract[\'id_col\'])\n    y = frame[contract[\'targets\']].to_numpy(float)\n    if not (np.isnan(y) | (y == 0) | (y == 1)).all():\n        raise ValueError(\'train.csv expert targets must be binary or missing\')\n    if report_path is not None:\n        report_frame = pd.read_csv(report_path, dtype={cfg[\'id_col\']: str})\n        report = aligned(report_frame, ids, contract)\n    else:\n        report = np.full_like(y, np.nan)\n        if cfg[\'auxiliary\'] != \'none\':\n            raise ValueError(\'Provide --report-labels for auxiliary training or set auxiliary=none\')\n    groups, caveat = load_groups(groups_path, ids, cfg[\'id_col\'])\n    fold, audit = make_folds(ids, y, groups, cfg[\'folds\'], cfg[\'fold_seed\'])\n    signature = {\'config\': cfg, \'schema\': contract, \'train_csv\': sha(root/\'train.csv\'),\n                 \'reports\': sha(report_path) if report_path else None,\n                 \'groups\': sha(groups_path) if groups_path else None, \'code\': code_hash()}\n    if (work/\'preparation.json\').exists():\n        old = json.loads((work/\'preparation.json\').read_text())\n        if old[\'signature\'] != signature:\n            raise ValueError(\'Preparation differs from existing work directory; use a new work directory\')\n    atomic_json(work/\'config.json\', cfg)\n    atomic_json(work/\'schema.json\', contract)\n    atomic_csv(work/\'expert_labels.csv\', frame[[cfg[\'id_col\']]+contract[\'targets\']])\n    reports = pd.DataFrame(report, columns=contract[\'targets\'])\n    reports.insert(0, cfg[\'id_col\'], ids)\n    atomic_csv(work/\'report_labels.csv\', reports)\n    atomic_csv(work/\'folds.csv\', pd.DataFrame({cfg[\'id_col\']: ids, \'group_id\': groups, \'fold\': fold}))\n    audit[\'grouping_caveat\'] = caveat\n    atomic_json(work/\'fold_audit.json\', audit)\n    cache = build_cache(root, \'train\', ids, cfg, work/\'cache\')\n    receipt = {\'signature\': signature, \'fold_audit\': audit, \'studies\': len(ids),\n               \'missing_images\': cache[\'missing_studies\'],\n               \'cache_manifest_sha256\': sha(work/\'cache/manifest.json\')}\n    atomic_json(work/\'preparation.json\', receipt)\n    return receipt\n\n\ndef _atomic_checkpoint(path, state):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(\'.tmp\')\n    torch.save(state, temp)\n    os.replace(temp, path)\n\n\ndef finite_optimizer_step(model, optimizer, scaler, max_norm):\n    """Let GradScaler reject overflow before clipping; never update invalid grads."""\n    scaler.unscale_(optimizer)\n    grads = [p.grad for p in model.parameters() if p.grad is not None]\n    finite = all(torch.isfinite(g).all().item() for g in grads)\n    if not finite:\n        if not scaler.is_enabled():\n            raise FloatingPointError(\'Nonfinite gradients in full precision\')\n        # unscale_ recorded the overflow. step skips the optimizer, update lowers\n        # the scale. The caller retries the SAME batch, preserving its exposure.\n        scaler.step(optimizer)\n        scaler.update()\n        optimizer.zero_grad(set_to_none=True)\n        return False\n    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm, error_if_nonfinite=True)\n    scaler.step(optimizer)\n    scaler.update()\n    return True\n\n\ndef _predict(model, dataset, indices, device, batch_size, prior):\n    predictions, available = [], []\n    model.eval()\n    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False, collate_fn=collate)\n    with torch.inference_mode():\n        for batch in loader:\n            batch = to_device(batch, device)\n            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                logits, has_images = model(batch)\n            p = logits.float().sigmoid().cpu().numpy()\n            present = has_images.cpu().numpy()\n            p[~present] = prior\n            predictions.append(p)\n            available.extend(present.tolist())\n    return np.concatenate(predictions), available\n\n\ndef train(work, fold, seed=None, arm=None):\n    work = Path(work)\n    start = time.monotonic()\n    cfg = json.loads((work/\'config.json\').read_text())\n    if seed is not None:\n        cfg[\'seed\'] = seed\n    if arm is not None:\n        cfg[\'auxiliary\'] = arm\n    if cfg[\'auxiliary\'] not in (\'none\', \'raw\', \'platt\'):\n        raise ValueError(\'Unknown supervision arm\')\n    contract = json.loads((work/\'schema.json\').read_text())\n    split = pd.read_csv(work/\'folds.csv\', dtype={cfg[\'id_col\']: str, \'group_id\': str})\n    ids, groups = check_ids(split, cfg[\'id_col\']), split.group_id.to_numpy()\n    if fold not in set(split.fold):\n        raise ValueError(\'Requested fold is absent\')\n    y = aligned(pd.read_csv(work/\'expert_labels.csv\', dtype={cfg[\'id_col\']: str}), ids, contract)\n    report = aligned(pd.read_csv(work/\'report_labels.csv\', dtype={cfg[\'id_col\']: str}), ids, contract)\n    tr, va = np.flatnonzero(split.fold.to_numpy() != fold), np.flatnonzero(split.fold.to_numpy() == fold)\n    if set(groups[tr]) & set(groups[va]):\n        raise ValueError(\'Group leakage in supplied folds\')\n    values, aux_mask, weights, policies = supervision(y, report, groups, tr, cfg, contract[\'targets\'], ids)\n    dataset = CachedStudies(work/\'cache\', ids, training=True, dropout=cfg[\'series_dropout\'])\n    if dataset.manifest[\'contract\'] != transform_contract(cfg):\n        raise ValueError(\'Cache preprocessing/runtime contract mismatch\')\n    present = np.array([dataset.manifest[\'entries\'][u][\'series\'] > 0 for u in ids])\n    expert_mask = np.isfinite(y) & present[:, None]\n    aux_mask &= present[:, None]\n    expert_rows = tr[expert_mask[tr].any(1)]\n    aux_rows = tr[(~expert_mask[tr].any(1)) & aux_mask[tr].any(1)]\n    if not len(expert_rows) and not len(aux_rows):\n        raise ValueError(\'No usable supervised training studies\')\n    prior = (np.nansum(y[tr], axis=0)+1)/(np.isfinite(y[tr]).sum(0)+2)\n    out = work/\'runs\'/f\'seed{cfg["seed"]}\'/cfg[\'auxiliary\']/f\'fold{fold}\'\n    out.mkdir(parents=True, exist_ok=True)\n    checkpoint_path = out/\'checkpoint.pt\'\n    binding = {\'config\': cfg, \'schema\': contract, \'fold\': int(fold), \'code\': code_hash(),\n               \'folds\': sha(work/\'folds.csv\'), \'expert\': sha(work/\'expert_labels.csv\'),\n               \'report\': sha(work/\'report_labels.csv\'), \'cache\': sha(work/\'cache/manifest.json\')}\n    signature = fingerprint(binding)\n    device = device_for(cfg[\'device\'])\n    torch.manual_seed(cfg[\'seed\'])\n    if device.type == \'cuda\':\n        torch.cuda.manual_seed_all(cfg[\'seed\'])\n    torch.backends.cudnn.benchmark = False\n    saved = torch.load(checkpoint_path, map_location=\'cpu\', weights_only=True) if checkpoint_path.exists() else None\n    if saved and saved[\'signature\'] != signature:\n        raise ValueError(\'Checkpoint is bound to different data/configuration/code\')\n    model = KneeModel(cfg, len(contract[\'targets\']), initialize=saved is None,\n                      encoder_spec=saved[\'encoder_spec\'] if saved else None).to(device)\n    encoder_ids = {id(p) for p in model.encoder.parameters()}\n    optimizer = torch.optim.AdamW([\n        {\'params\': [p for p in model.parameters() if p.requires_grad and id(p) in encoder_ids], \'lr\': cfg[\'encoder_lr\']},\n        {\'params\': [p for p in model.parameters() if p.requires_grad and id(p) not in encoder_ids], \'lr\': cfg[\'head_lr\']}],\n        weight_decay=cfg[\'weight_decay\'])\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg[\'epochs\'])\n    scaler = torch.amp.GradScaler(device.type, init_scale=1024., enabled=device.type == \'cuda\')\n    history, start_epoch = [], 0\n    if saved:\n        model.load_state_dict(saved[\'model\'], strict=True)\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        torch.set_rng_state(saved[\'rng\'])\n        if device.type == \'cuda\' and saved.get(\'cuda_rng\') is not None:\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        start_epoch, history = saved[\'epoch\'], saved[\'history\']\n    y_t = torch.tensor(y, dtype=torch.float32)\n    aux_t = torch.tensor(values)\n    em_t, am_t = torch.tensor(expert_mask), torch.tensor(aux_mask)\n    weight_t = torch.tensor(weights, device=device)\n    id_index = {uid: i for i, uid in enumerate(ids)}\n    for epoch in range(start_epoch, cfg[\'epochs\']):\n        rng = np.random.default_rng(cfg[\'seed\']+epoch)\n        # Keep every expert study and every eligible report-only study; supplement expert exposure.\n        extra = 0 if not len(expert_rows) else max(0, int(np.ceil(\n            len(aux_rows)*cfg[\'expert_fraction\']/(1-cfg[\'expert_fraction\'])))-len(expert_rows))\n        indices = np.concatenate([expert_rows, aux_rows, rng.choice(expert_rows, extra, replace=True)\n                                  if extra else np.array([], int)])\n        rng.shuffle(indices)\n        loader = DataLoader(Subset(dataset, indices.tolist()), batch_size=cfg[\'batch_size\'],\n                            shuffle=False, collate_fn=collate)\n        model.train()\n        running, steps, overflow_retries = 0., 0, 0\n        for batch in loader:\n            index = torch.tensor([id_index[u] for u in batch[\'ids\']])\n            batch = to_device(batch, device)\n            for attempt in range(12):\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                    logits, has_images = model(batch)\n                expert_loss = masked_loss(logits, y_t[index].to(device),\n                                          em_t[index].to(device) & has_images[:, None], cfg)\n                auxiliary_loss = masked_loss(logits, aux_t[index].to(device),\n                                             am_t[index].to(device) & has_images[:, None], cfg,\n                                             auxiliary=True, weights=weight_t)\n                loss = expert_loss+auxiliary_loss\n                if not torch.isfinite(loss):\n                    raise ValueError(f\'Nonfinite training loss for {batch["ids"]}\')\n                scaler.scale(loss).backward()\n                if finite_optimizer_step(model, optimizer, scaler, cfg[\'grad_clip\']):\n                    break\n                overflow_retries += 1\n                print(f\'AMP overflow: retry {attempt+1}/12, scale={scaler.get_scale()}, \'\n                      f\'studies={batch["ids"]}\', flush=True)\n            else:\n                raise FloatingPointError(f\'Persistent gradient overflow for {batch["ids"]}\')\n            running += loss.item()\n            steps += 1\n            if steps % 100 == 0:\n                print(f\'Fold {fold}, epoch {epoch+1}/{cfg["epochs"]}, batch {steps}/{len(loader)}, \'\n                      f\'loss={running/steps:.5f}, AMP retries={overflow_retries}\', flush=True)\n        scheduler.step()\n        history.append({\'epoch\': epoch+1, \'loss\': running/max(steps, 1), \'steps\': steps,\n                        \'row_instances\': len(indices), \'overflow_retries\': overflow_retries})\n        state = {\'model\': model.state_dict(), \'encoder_spec\': model.encoder.spec, \'config\': cfg,\n                 \'schema\': contract, \'preprocessing\': dataset.manifest[\'contract\'],\n                 \'prior\': prior.tolist(), \'signature\': signature, \'binding\': binding,\n                 \'train_ids\': [ids[i] for i in tr], \'val_ids\': [ids[i] for i in va],\n                 \'train_groups\': sorted(set(groups[tr])), \'val_groups\': sorted(set(groups[va])),\n                 \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                 \'scaler\': scaler.state_dict(), \'epoch\': epoch+1, \'history\': history,\n                 \'rng\': torch.get_rng_state(),\n                 \'cuda_rng\': torch.cuda.get_rng_state_all() if device.type == \'cuda\' else None,\n                 \'policies\': policies}\n        _atomic_checkpoint(checkpoint_path, state)\n        print(f\'Fold {fold}, {cfg["auxiliary"]}, epoch {epoch+1}/{cfg["epochs"]}: loss={history[-1]["loss"]:.5f}\', flush=True)\n    dataset.training = False\n    p, has_images = _predict(model, dataset, va.tolist(), device, cfg[\'batch_size\'], prior)\n    write_predictions(out/\'oof.csv\', [ids[i] for i in va], p, contract)\n    receipt = {\'status\': \'FOLD_COMPLETE\', \'seed\': cfg[\'seed\'], \'arm\': cfg[\'auxiliary\'], \'fold\': int(fold),\n               \'checkpoint_sha256\': sha(checkpoint_path), \'oof_sha256\': sha(out/\'oof.csv\'),\n               \'signature\': signature, \'binding\': binding, \'metrics\': metrics(y[va], p, contract[\'targets\']),\n               \'train_ids\': [ids[i] for i in tr], \'val_ids\': [ids[i] for i in va],\n               \'fallback_ids\': [ids[i] for i, ok in zip(va, has_images) if not ok],\n               \'epoch_selection\': \'fixed final epoch; outer labels used only after training\',\n               \'policies\': policies, \'history\': history, \'runtime_seconds\': time.monotonic()-start}\n    atomic_json(out/\'receipt.json\', receipt)\n    return receipt\n\n\ndef merge(work, seed, arm):\n    work = Path(work)\n    contract = json.loads((work/\'schema.json\').read_text())\n    split = pd.read_csv(work/\'folds.csv\', dtype={contract[\'id_col\']: str})\n    ids = check_ids(split, contract[\'id_col\'])\n    parts, checkpoint_hashes, common_binding = [], [], None\n    root = work/\'runs\'/f\'seed{seed}\'/arm\n    for fold in sorted(split.fold.unique()):\n        directory = root/f\'fold{fold}\'\n        receipt = json.loads((directory/\'receipt.json\').read_text())\n        if receipt[\'seed\'] != seed or receipt[\'arm\'] != arm or receipt[\'fold\'] != fold:\n            raise ValueError(\'Shard identity mismatch\')\n        if receipt[\'binding\'][\'folds\'] != sha(work/\'folds.csv\') or receipt[\'oof_sha256\'] != sha(directory/\'oof.csv\') or receipt[\'checkpoint_sha256\'] != sha(directory/\'checkpoint.pt\'):\n            raise ValueError(\'Shard provenance/checksum mismatch\')\n        binding = {k: v for k, v in receipt[\'binding\'].items() if k != \'fold\'}\n        if common_binding is not None and binding != common_binding:\n            raise ValueError(\'Shards have different training configurations/data\')\n        common_binding = binding\n        part = pd.read_csv(directory/\'oof.csv\', dtype={contract[\'id_col\']: str})\n        expected = split.loc[split.fold == fold, contract[\'id_col\']].tolist()\n        aligned(part, expected, contract, probabilities=True)\n        if set(receipt[\'train_ids\']) != set(ids)-set(expected) or set(receipt[\'val_ids\']) != set(expected):\n            raise ValueError(\'Shard held-out provenance mismatch\')\n        parts.append(part)\n        checkpoint_hashes.append(receipt[\'checkpoint_sha256\'])\n    p = aligned(pd.concat(parts), ids, contract, probabilities=True)\n    write_predictions(root/\'oof.csv\', ids, p, contract)\n    y = aligned(pd.read_csv(work/\'expert_labels.csv\', dtype={contract[\'id_col\']: str}), ids, contract)\n    result = {\'status\': \'ALL_FOLDS_COMPLETE\', \'metrics\': metrics(y, p, contract[\'targets\']),\n              \'oof_sha256\': sha(root/\'oof.csv\'), \'checkpoint_hashes\': checkpoint_hashes,\n              \'cohort\': \'expert-label evaluation; unknown cells excluded; no target silently dropped\'}\n    atomic_json(root/\'oof_receipt.json\', result)\n    return result\n\n\ndef infer(root, checkpoints, output, cache_dir, device=\'auto\', baseline=None, blend_weight=None,\n          encoder_code=None, max_seconds=28800):\n    output, root = Path(output), Path(root)\n    start = time.monotonic()\n    if max_seconds <= 0:\n        raise ValueError(\'max_seconds must be positive\')\n    if baseline is not None and (blend_weight is None or not 0 < blend_weight <= 1):\n        raise ValueError(\'Baseline blending requires explicit --blend-weight in (0,1]\')\n    if baseline is not None and Path(baseline).resolve() in (\n        output.resolve(), output.with_name(output.stem+\'_model.csv\').resolve()):\n        raise ValueError(\'Baseline input must be separate from output files\')\n    paths = list(dict.fromkeys(str(Path(p).resolve()) for p in checkpoints))\n    if len(paths) != len(checkpoints):\n        raise ValueError(\'Repeated checkpoints would silently change ensemble weights\')\n    total, manifest, rows, contract, receipts = None, None, None, None, []\n    baseline_values = None\n    actual_device = device_for(device)\n    skipped = []\n    for path in paths:\n        if receipts and time.monotonic()-start + max(r[\'runtime_seconds\'] for r in receipts)*1.25 > max_seconds:\n            skipped = paths[len(receipts):]\n            break\n        arm_start = time.monotonic()\n        state = torch.load(path, map_location=\'cpu\', weights_only=True)\n        cfg, stored = dict(state[\'config\']), state[\'schema\']\n        if state[\'epoch\'] != cfg[\'epochs\']:\n            raise ValueError(\'Checkpoint has not completed its declared training schedule\')\n        if state[\'signature\'] != fingerprint(state[\'binding\']) or state[\'binding\'][\'code\'] != code_hash():\n            raise ValueError(\'Checkpoint provenance/runtime code differs; use its matching V16 package\')\n        if encoder_code:\n            cfg[\'dinov2_repo\'] = encoder_code\n        live = schema(root, stored[\'id_col\'])\n        if set(live[\'targets\']) != set(stored[\'targets\']):\n            raise ValueError(\'Checkpoint target set differs from live schema\')\n        permutation = [stored[\'targets\'].index(t) for t in live[\'targets\']]\n        frame = read_studies(root, \'test\', live)\n        ids = check_ids(frame, live[\'id_col\'])\n        if state[\'preprocessing\'] != transform_contract(cfg):\n            raise ValueError(\'Inference decoder/runtime differs from saved training contract\')\n        if manifest is None:\n            manifest = {\'contract\': transform_contract(cfg)}\n            rows, contract = ids, live\n            if baseline is not None:\n                baseline_values = aligned(pd.read_csv(baseline, dtype={live[\'id_col\']: str}), ids, live, probabilities=True)\n        elif manifest[\'contract\'] != state[\'preprocessing\'] or rows != ids or contract != live:\n            raise ValueError(\'This ensemble requires matching preprocessing/schema/study contracts\')\n        model = KneeModel(cfg, len(stored[\'targets\']), initialize=False, encoder_spec=state[\'encoder_spec\']).to(actual_device)\n        model.load_state_dict(state[\'model\'], strict=True)\n        dataset = DicomStudies(root, ids, cfg)\n        p, present = _predict(model, dataset, list(range(len(ids))), actual_device, cfg[\'batch_size\'], np.array(state[\'prior\']))\n        p = p[:, permutation]\n        total = p.astype(np.float64) if total is None else total+p\n        receipts.append({\'checkpoint\': path, \'sha256\': sha(path),\n                         \'fallback_ids\': [u for u, ok in zip(ids, present) if not ok], \'encoder\': cfg[\'encoder\'],\n                         \'runtime_seconds\': time.monotonic()-arm_start})\n        atomic_json(Path(cache_dir)/f\'decode_audit_{len(receipts)}.json\', dataset.events)\n        # Bank only whole-cohort arms. Never mix partial per-study model sets.\n        banked = total/len(receipts)\n        if baseline is not None:\n            banked = (1-blend_weight)*baseline_values + blend_weight*banked\n        write_predictions(output, rows, banked, contract)\n        del model, state\n        if actual_device.type == \'cuda\':\n            torch.cuda.empty_cache()\n    if total is None:\n        raise ValueError(\'No checkpoints supplied\')\n    p = total / len(receipts)\n    raw_path = output.with_name(output.stem+\'_model.csv\')\n    write_predictions(raw_path, rows, p, contract)\n    if baseline is not None:\n        p = (1-blend_weight)*baseline_values + blend_weight*p\n    write_predictions(output, rows, p, contract)\n    receipt = {\'status\': \'INFERENCE_COMPLETE\', \'studies\': len(rows), \'schema\': contract,\n               \'checkpoints\': receipts, \'preprocessing\': manifest[\'contract\'],\n               \'blend\': \'fixed arithmetic mean; no inference-batch ranking\',\n               \'baseline\': {\'sha256\': sha(baseline), \'candidate_weight\': blend_weight} if baseline else None,\n               \'submission_sha256\': sha(output), \'model_predictions_sha256\': sha(raw_path),\n               \'skipped_checkpoints_for_budget\': skipped,\n               \'runtime_seconds\': time.monotonic()-start, \'max_seconds\': max_seconds,\n               \'measured_auc\': None, \'code_sha256\': code_hash()}\n    atomic_json(output.with_suffix(\'.receipt.json\'), receipt)\n    return receipt\n', encoding='utf-8')
(runtime/'v16'/'evaluation.py').write_text('"""Paired group-bootstrap evaluation; never projects a leaderboard score."""\nfrom pathlib import Path\nimport json\n\nimport numpy as np\nimport pandas as pd\n\nfrom .common import aligned, atomic_json, check_ids, sha\nfrom .validation import metrics\n\n\ndef evaluate(work, baseline, candidate, output, weight=1., bootstrap=2000, seed=1700):\n    if not 0 < weight <= 1 or bootstrap < 20:\n        raise ValueError(\'Invalid candidate weight or bootstrap count\')\n    work = Path(work)\n    contract = json.loads((work/\'schema.json\').read_text())\n    labels = pd.read_csv(work/\'expert_labels.csv\', dtype={contract[\'id_col\']: str})\n    ids = check_ids(labels, contract[\'id_col\'])\n    y = aligned(labels, ids, contract)\n    b = aligned(pd.read_csv(baseline, dtype={contract[\'id_col\']: str}), ids, contract, probabilities=True)\n    c = aligned(pd.read_csv(candidate, dtype={contract[\'id_col\']: str}), ids, contract, probabilities=True)\n    c = (1-weight)*b + weight*c\n    base_metrics, cand_metrics = metrics(y, b, contract[\'targets\']), metrics(y, c, contract[\'targets\'])\n    if not base_metrics[\'all_targets_scorable\']:\n        raise ValueError(\'Every target needs both expert classes; cannot compute full macro AUC\')\n    group_frame = pd.read_csv(work/\'folds.csv\', dtype={contract[\'id_col\']: str, \'group_id\': str})\n    group = group_frame.set_index(contract[\'id_col\']).loc[ids, \'group_id\'].to_numpy()\n    gold = np.isfinite(y).any(1)\n    y, b, c, group = y[gold], b[gold], c[gold], group[gold]\n    names = np.unique(group)\n    indices = [np.flatnonzero(group == g) for g in names]\n    rng, draws = np.random.default_rng(seed), []\n    for _ in range(bootstrap):\n        draw = np.concatenate([indices[i] for i in rng.integers(len(names), size=len(names))])\n        bm = metrics(y[draw], b[draw], contract[\'targets\'])[\'macro_auc\']\n        cm = metrics(y[draw], c[draw], contract[\'targets\'])[\'macro_auc\']\n        if bm is not None and cm is not None:\n            draws.append(cm-bm)\n    valid = len(draws) >= .8*bootstrap\n    result = {\'scope\': \'Paired development estimate against expert labels; no leaderboard projection\',\n              \'baseline_oof_provenance\': \'CSV inputs must be independently audited as held out; filename is not evidence\',\n              \'candidate_weight\': weight, \'baseline\': base_metrics, \'candidate\': cand_metrics,\n              \'delta\': cand_metrics[\'macro_auc\']-base_metrics[\'macro_auc\'],\n              \'delta_ci95\': np.quantile(draws, [.025, .975]).tolist() if valid else None,\n              \'bootstrap_requested\': bootstrap, \'bootstrap_valid\': len(draws),\n              \'interval_usable\': valid, \'groups_resampled\': len(names),\n              \'baseline_sha256\': sha(baseline), \'candidate_sha256\': sha(candidate)}\n    atomic_json(output, result)\n    return result\n', encoding='utf-8')
(runtime/'v16'/'config.json').write_text('{\n  "id_col": "StudyInstanceUID",\n  "series_col": "SeriesInstanceUID",\n  "image_size": 336,\n  "crop_mm": 140.0,\n  "offsets": [-1, 0, 1],\n  "max_centres": 24,\n  "max_series": 8,\n  "percentiles": [2.0, 98.0],\n  "encoder": "dinov2",\n  "encoder_path": "",\n  "dinov2_repo": "",\n  "cnn_arch": "convnext_tiny",\n  "unfreeze_last": 2,\n  "hidden": 128,\n  "encoder_chunk": 8,\n  "gradient_checkpointing": true,\n  "folds": 5,\n  "fold_seed": 1400,\n  "seed": 1400,\n  "epochs": 8,\n  "batch_size": 1,\n  "head_lr": 0.001,\n  "encoder_lr": 0.000008,\n  "weight_decay": 0.01,\n  "grad_clip": 1.0,\n  "expert_fraction": 0.1,\n  "auxiliary": "platt",\n  "silent_value": 0.5,\n  "aux_cap": 0.3,\n  "policy_bootstrap": 500,\n  "expert_loss": "bce",\n  "gamma_pos": 0.0,\n  "gamma_neg": 2.0,\n  "series_dropout": 0.1,\n  "device": "auto"\n}\n', encoding='utf-8')
sys.path.insert(0,str(runtime))


In [ ]:
import shutil, pandas as pd
from v16.common import config
from v16.runner import prepare, train, finite_optimizer_step
from v16.model import KneeModel, masked_loss
from v16.data import collate, to_device
roots=[]; models=[]; reports=[]
for folder, dirs, names in os.walk('/kaggle/input'):
    if 'train.csv' in names and 'sample_submission.csv' in names:
        roots.append(Path(folder))
    if 'config.json' in names and ('pytorch_model.bin' in names or 'model.safetensors' in names):
        models.append(Path(folder))
    if 'llm_labels_v2.csv' in names:
        reports.append(Path(folder)/'llm_labels_v2.csv')
    dirs[:]=[d for d in dirs if d not in ('train_series','test_series')]
assert len(roots)==len(models)==len(reports)==1, (roots,models,reports)
cfg=config(runtime/'v16/config.json')
cfg.update(encoder_path=str(models[0]),image_size=224,max_centres=8,max_series=3,
           encoder_chunk=4,device='cuda')
# Check the actual allocated GPU BEFORE spending hours on image preparation.
targets=list(pd.read_csv(roots[0]/'sample_submission.csv',nrows=0).drop(columns=cfg['id_col']).columns)
probe=KneeModel(cfg,len(targets)).cuda().train()
optimizer=torch.optim.AdamW([p for p in probe.parameters() if p.requires_grad],lr=1e-5)
scaler=torch.amp.GradScaler('cuda',init_scale=1024.)
for step in range(20):
    batch=collate([{'uid':'numerical-probe','series':[
        {'x':torch.rand(cfg['max_centres'],3,cfg['image_size'],cfg['image_size']),
         'position':torch.linspace(0,1,cfg['max_centres']),'meta':torch.zeros(6)}
        for _ in range(cfg['max_series'])]}])
    batch=to_device(batch,torch.device('cuda'))
    for attempt in range(12):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):
            logits,_=probe(batch)
        truth=torch.full_like(logits,float(step%2))
        loss=masked_loss(logits,truth,torch.ones_like(logits,dtype=torch.bool),cfg)
        assert torch.isfinite(loss), 'GPU preflight loss is nonfinite'
        scaler.scale(loss).backward()
        if finite_optimizer_step(probe,optimizer,scaler,cfg['grad_clip']): break
    else: raise RuntimeError('GPU preflight could not recover from overflow')
    assert all(torch.isfinite(p).all() for p in probe.parameters())
print('GPU stability preflight passed: 20 optimizer steps',flush=True)
del probe,optimizer,scaler,batch,logits,loss
torch.cuda.empty_cache()
work=Path('/tmp/v16_work'); work.mkdir(exist_ok=True)
output=Path('/kaggle/working/v16_output'); output.mkdir(exist_ok=True)
runs=output/'runs'; runs.mkdir(exist_ok=True)
if not (work/'runs').exists():
    (work/'runs').symlink_to(runs,target_is_directory=True)
n=len(pd.read_csv(roots[0]/'train.csv'))
bound=n*cfg['max_series']*cfg['max_centres']*len(cfg['offsets'])*cfg['image_size']**2*2
free=shutil.disk_usage(work).free
assert free>bound*1.1+5*2**30, f'Cache needs up to {bound/2**30:.1f} GiB plus headroom; free {free/2**30:.1f}'
print('Preparing all training studies; scratch cache is excluded from saved output',flush=True)
receipt=prepare(roots[0],work,cfg,reports[0])
for path in work.iterdir():
    if path.is_file(): shutil.copy2(path,output/path.name)
shutil.copy2(work/'cache/manifest.json',output/'cache_manifest.json')
assert not receipt['missing_images'], 'Review missing training image diagnostics before training'
print('Training fold 0; study-only grouping, patient separation unverified',flush=True)
result=train(work,0,seed=1400,arm='platt')
print(json.dumps(result,indent=2),flush=True)
